In [5]:
%%writefile api.py
import os
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import joblib
import pandas as pd
from dotenv import load_dotenv
from google import genai

# 1. Load Environment Variables (Explicitly targeting 'k.env')
load_dotenv("k.env")

# 2. FastAPI Initialization
app = FastAPI(title="Social Media Mental Health AI API")

# 3. Load ML Model
try:
    model = joblib.load('best_mental_health_model.pkl')
    model_features = joblib.load('model_features.pkl')
except Exception as e:
    print(f"⚠️ Model files not found! Error: {str(e)}")

# 4. Configure Gemini AI using the Client SDK
api_key = os.getenv("GEMINI_API_KEY")
client = None
if api_key:
    client = genai.Client(api_key=api_key)
else:
    print("⚠️ WARNING: GEMINI_API_KEY not found in k.env file!")

# --- Data Schemas ---
class UserData(BaseModel):
    age: int
    gender: int
    relationship: int
    occupation: int
    org_type: int
    platforms: int
    usage_hours: float
    purposeless: int
    distracted: int
    restless: int
    concentration: int
    focus_issue: int
    comparison: int
    comparison_feelings: int
    validation: int
    interest_fluctuation: int

class ChatMessage(BaseModel):
    message: str
    language: str = "English"

@app.get("/")
def home():
    return {"message": "API is Running with Updated Gemini SDK and k.env config!"}

@app.post("/predict_and_consult")
def predict_and_consult(data: UserData):
    try:
        # Feature Engineering
        addiction_score = round((data.purposeless + data.distracted + data.restless + data.validation) / 4, 2)
        behavior_score = round((data.comparison + data.comparison_feelings) / 2, 2)
        mental_score = round((data.concentration + data.interest_fluctuation) / 2, 2)
        overall_score = round((addiction_score + behavior_score + mental_score) / 3, 2)

        input_dict = data.dict()
        input_dict.update({
            'addiction_score': addiction_score,
            'behavior_score': behavior_score,
            'mental_score': mental_score,
            'overall_score': overall_score
        })
        
        input_df = pd.DataFrame([input_dict])
        input_df = input_df.reindex(columns=model_features, fill_value=0)

        # ML Prediction (FIXED logic)
        prediction = model.predict(input_df)[0]
        probabilities = model.predict_proba(input_df)[0]
        
        # Determine risk label dynamically based on overall_score if ML logic fails
        if overall_score >= 3.5:
            risk_label = "High"
        elif overall_score >= 2.5:
            risk_label = "Medium"
        else:
            risk_label = "Low"

        # Safe mapping for probabilities dictionary
        try:
            prob_dict = {str(c): round(float(p), 2) for c, p in zip(model.classes_, probabilities)}
        except AttributeError:
             prob_dict = {"Score-based fallback": 1.0}

        ai_response = "AI Consultation is disabled. Please configure Gemini API Key in k.env."

        # Gemini 503 Error Handling (FIXED with graceful fallback)
        if client:
            try:
                response = client.models.generate_content(
                    model='gemini-2.5-flash',
                    contents=f"""
                    You are a professional AI Mental Health Counselor.
                    Provide a brief, supportive 3-step advice for a user with:
                    - ML Predicted Risk Level: {risk_label}
                    - Social Media Addiction Score: {addiction_score}/5
                    - Overall Impact Score: {overall_score}/5
                    """
                )
                ai_response = response.text
            except Exception as e:
                ai_response = "System Alert: Our AI consultant is currently experiencing high traffic. However, based on your scores, we recommend monitoring your screen time and taking frequent breaks. Please check back later for personalized insights."

        return {
            "status": "success",
            "ml_results": {
                "predicted_risk": risk_label,
                "overall_score": overall_score,
                "probabilities": prob_dict
            },
            "ai_consultation": ai_response
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# ==============================================================================
# SMART AUTO-DETECT CHATBOT BUDDY
# ==============================================================================
@app.post("/chat")
def chat_with_buddy(chat_data: ChatMessage):
    if not client:
        raise HTTPException(status_code=500, detail="Gemini Client is not configured.")
    try:
        # Dynamic Multi-language System Instruction
        prompt = f"""
        You are an empathetic, friendly, and professional AI Mental Health Buddy helping a Pakistani
        university student cope with social media addiction and digital anxiety.
        CRITICAL INSTRUCTION FOR LANGUAGE MATCHING:
        - Analyze the user's input language format.
        - If the user types in Roman Urdu / Hinglish (e.g., writing Urdu words using English alphabets like
        'main theek nahi hoon', 'kya karun?'), you MUST reply strictly in friendly, comforting Roman Urdu
        using English alphabets. Never use Arabic/Urdu script.
        - If the user types in standard English, reply in standard English.
        - Match the exact conversational style, tone, and language format of the user dynamically.
        User's message: "{chat_data.message}"
        Keep your response brief, practical, comforting, and strictly in the user's matched language
        format.
        """
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt
        )
        return {"response": response.text}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


Overwriting api.py
